# 31 — W5 B2 stage: S-DPO refinement of the responder LLM (CONDITIONAL)

Per RecSys_Challenge_Plan §6.3 row B2 + §6.4. **Conditional** stage: only run if W4 KTO format compliance < 70%. If W4 already passes the gate, **skip this notebook entirely** and proceed to W6 (Rank-GRPO).

## Why S-DPO is needed (plan §6.3)

If W4 KTO didn't reliably teach the model to emit the envelope (e.g., format compliance ~50-65%), S-DPO with explicit (chosen=well-formed, rejected=degraded) preference pairs gives a stronger format-reinforcement signal. We use **vanilla TRL DPOTrainer** on a 1-positive-N-negatives → N-pairs expansion of the GPA POS rows. This loses the Plackett-Luce softmax (the paper's S-DPO formulation) but is upstream-trainable in TRL — the data is `(prompt, chosen, rejected)` triples.

## Pipeline

1. Mount Drive, install deps, HF auth.
2. Gate-check: confirm W4 KTO ran AND format compliance < 70%. Abort if not.
3. Pytest pre-flight (all module tests).
4. Build the S-DPO parquet on the fly: `build_reward_dataset → augment_envelope → build_sdpo_dataset`.
5. TRL DPOTrainer on Qwen-2.5-7B + W4 KTO adapter as starting LoRA, fresh r=32 LoRA layered on top.
6. Format-compliance check (same chat-template + r_format gate as W4).

## Compute (plan §6.3 B2 row)

- ~6 A100-hr for ~30k POS turns × 4 negs = ~120k DPO pairs × 1 epoch.
- Same OOM ladder as W4 if VRAM gets tight.

**Reminder for follow-up scaling work:** if compute is constrained, reduce `n_sessions` in cell 6 (data build) — the data scales linearly with positive turns. Halving to 50% data ≈ 3 A100-hr.

## ⚠️ KNOWN PLAN-DEVIATION (W5 review P1 #10)

Plan §6.4 specifies **6 negatives per positive turn**: 3 hard-track BM25 negs (paired with on-policy Qwen responses) + 2 response-only mutations + 1 GPA-DOES_NOT_MOVE. This implementation ships **only 4 negatives**: 3 response-only perturbations (drop_track_name, inject_banned, truncate_5) + 1 cross-session GPA NEG (with drop_why fallback).

The **3 hard-track on-policy negatives are deferred** — generating them requires running Qwen-7B inference on ~30k × 3 = 90k extra hard-track candidates (~2 A100-hr). The 4 cheap negatives mainly target *format* and *vocabulary* signals; they do NOT teach **track-grounding** (the "given the wrong track, the response should differ from gold" axis). If the W5 format gate misses, the next thing to try is the deferred hard-track extension, NOT more S-DPO epochs.


In [ ]:
# 1) GPU check.
!nvidia-smi | head -20

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026
!git log -1 --pretty=format:'commit:  %h%nsubject: %s'

In [ ]:
# 2b) Mount Drive + persistent caches.
import os, shutil
from google.colab import drive

try: drive.mount('/content/drive')
except Exception as e:
    print(f'first mount attempt failed: {e}; retrying ...')
    try: drive.flush_and_unmount()
    except Exception: pass
    drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = '/content/drive/MyDrive/recsys2026-cache'
for d in [f'{DRIVE_BASE}/hf_datasets', f'{DRIVE_BASE}/experiments_cache',
          f'{DRIVE_BASE}/sdpo_runs']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_DATASETS_CACHE'] = f'{DRIVE_BASE}/hf_datasets'
%env HF_DATASETS_CACHE={DRIVE_BASE}/hf_datasets

EXPECTED_CACHE = '/content/recsys2026/music-crs-baselines/experiments/cache'
os.makedirs(os.path.dirname(EXPECTED_CACHE), exist_ok=True)
if os.path.exists(EXPECTED_CACHE) and not os.path.islink(EXPECTED_CACHE):
    shutil.rmtree(EXPECTED_CACHE)
if not os.path.islink(EXPECTED_CACHE):
    os.symlink(f'{DRIVE_BASE}/experiments_cache', EXPECTED_CACHE)

In [ ]:
# 3) HF auth — abort if missing (same pattern as W4 fix).
from google.colab import userdata
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN
    from huggingface_hub import whoami
    user = whoami(token=HF_TOKEN)
    print(f'✓ HF auth ok — logged in as {user["name"]}')
except (userdata.SecretNotFoundError, Exception) as e:
    print(f'❌ HF auth failed: {e!r}')
    raise SystemExit('HF_TOKEN required.')

In [ ]:
# 4) Gate-check: W5 is CONDITIONAL — only run if W4 KTO format < 70%.
# Reads the W4 gate_result.json from Drive; aborts if W4 already passed.
import json
from pathlib import Path

# Find the most recent W4 gate_result.json in kto_runs/
kto_runs = Path(f'{DRIVE_BASE}/kto_runs')
candidates = list(kto_runs.rglob('gate_result.json'))
if not candidates:
    print('⚠️  No W4 KTO gate_result.json found in Drive.')
    print(f'    Looked under: {kto_runs}/<run_name>/gate_result.json')
    print('    Either W4 has not run yet, OR you want to run W5 standalone.')
    print('    Set W5_FORCE = True below to override the gate check.')
    W5_FORCE = False  # ← change to True to bypass
    if not W5_FORCE:
        raise SystemExit('W5 aborted — no W4 result and W5_FORCE=False.')
    W4_FORMAT = None
    KTO_ADAPTER = None
else:
    latest = max(candidates, key=lambda p: p.stat().st_mtime)
    with latest.open() as f:
        w4 = json.load(f)
    W4_FORMAT = w4.get('format_compliance_strict', w4.get('format_compliance', 0.0))
    W4_RUN = w4.get('run_name')
    KTO_ADAPTER = w4.get('hub_model')
    print(f'W4 latest run:    {W4_RUN}')
    print(f'W4 strict format: {W4_FORMAT:.1%}')
    print(f'W4 hub model:     {KTO_ADAPTER}')
    if W4_FORMAT >= 0.70:
        print(f'\n✓ W4 format ≥ 70% — W5 is NOT NEEDED. Skip to W6 (Rank-GRPO).')
        print('  If you want to run S-DPO anyway (e.g., to ablate format vs preference\n'
              '  signals), set W5_FORCE = True below.')
        W5_FORCE = False
        if not W5_FORCE:
            raise SystemExit('W5 not needed — W4 format passed.')
    else:
        print(f'\n→ W4 format < 70% — W5 S-DPO RECOMMENDED for format reinforcement.')

In [ ]:
# 5) Install deps + pytest pre-flight.
!pip install -q --upgrade transformers datasets 'pandas<3.0' tqdm omegaconf
!pip install -q --upgrade 'trl>=0.12.0' 'peft>=0.7.0' trackio accelerate
!python -c 'import torch, transformers, trl, peft; print("torch", torch.__version__, "trl", trl.__version__, "peft", peft.__version__)'

# Pytest pre-flight — confirm augmenter / build_trl_datasets / build_sdpo_dataset all green.
# W5 P1 #4 fix: include test_state_tracker / test_cmqr / test_pro_rank
# because config 210 (dev-eval) uses use_state_tracker=true, use_cmqr=true,
# and reranker_type=pro_rank. Same gap caught in W4 review.
!cd /content/recsys2026 && python -m pytest \
    tests/test_reward_fns.py \
    tests/test_augment_envelope.py \
    tests/test_build_trl_datasets.py \
    tests/test_build_sdpo_dataset.py \
    tests/test_state_tracker.py \
    tests/test_cmqr.py \
    tests/test_pro_rank.py \
    -q


In [ ]:
# 6) Build the S-DPO parquet on the fly (idempotent — skips if already built).
# Pipeline: build_reward_dataset → augment_envelope → build_sdpo_dataset.
#
# SCALING NOTE: to reduce wallclock, set N_SESSIONS lower (e.g., 5000 for
# half the cost). Linear time scaling. Plan §6.4 budgets ~30k POS turns ≈
# ~120k pairs at full data; halving = ~60k pairs ≈ ~3 A100-hr.
N_SESSIONS = 15000  # ← tune this knob for compute/quality tradeoff

REWARD = '/content/recsys2026/data/reward_train.parquet'
ENVELOPE = '/content/recsys2026/data/reward_train_envelope.parquet'
SDPO = '/content/recsys2026/data/trl/sdpo.parquet'

if not Path(REWARD).exists():
    !cd /content/recsys2026 && python scripts/build_reward_dataset.py --n-sessions {N_SESSIONS} --out {REWARD}
else:
    print(f'reusing existing {REWARD}')

if not Path(ENVELOPE).exists():
    !cd /content/recsys2026 && python scripts/augment_envelope.py --in {REWARD} --out {ENVELOPE}
else:
    print(f'reusing existing {ENVELOPE}')

if not Path(SDPO).exists():
    !cd /content/recsys2026 && python scripts/build_sdpo_dataset.py --in {ENVELOPE} --out {SDPO}
else:
    print(f'reusing existing {SDPO}')

import pandas as pd
d = pd.read_parquet(SDPO)
print(f'\nS-DPO dataset: {len(d):,} pairs')
print(f'columns: {list(d.columns)}')
print(f'\nneg-type breakdown:')
print(d['neg_type'].value_counts())

In [ ]:
# 7) Schema validation + train/eval split.
from datasets import Dataset
ds = Dataset.from_pandas(d, preserve_index=False)
# Drop the diagnostic neg_type column before training (TRL only needs prompt/chosen/rejected)
trl_cols = ['prompt', 'chosen', 'rejected', 'split']
ds = ds.select_columns([c for c in trl_cols if c in ds.column_names])

# Schema sanity
assert set(ds.column_names) >= {'prompt', 'chosen', 'rejected'}, ds.column_names
for col in ['prompt', 'chosen', 'rejected']:
    assert isinstance(ds[0][col], str)
print(f'✓ schema (prompt, chosen, rejected: str) PASS')

# Train/eval split (90/10)
split = ds.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split['train'], split['test']
print(f'train: {len(train_ds):,}  eval: {len(eval_ds):,}')

# Verify no degenerate pairs
n_degen = sum(1 for r in ds if r['chosen'] == r['rejected'])
assert n_degen == 0, f'{n_degen} degenerate (chosen==rejected) pairs in dataset'
print('✓ no degenerate pairs')

In [ ]:
# 8) Trackio init — same group="b-stage" as W4 so dashboards line up.
from datetime import date
import trackio

RUN_NAME = f'b2-sdpo-qwen7b-{date.today().isoformat()}'
TRACKIO_OK = True
try:
    trackio.init(
        project='recsys2026',
        name=RUN_NAME,
        # group='b-stage' — removed (current trackio doesn't accept group kwarg; set via config below if desired)
        config={
            'model': 'Qwen/Qwen2.5-7B-Instruct',
            'method': 'S-DPO (vanilla TRL DPOTrainer; 1-pos-N-neg expanded to pairs)',
            'starting_adapter': KTO_ADAPTER or 'base (no W4)',
            'dataset_size': len(train_ds),
            'eval_size': len(eval_ds),
            'lora_r': 32, 'lora_alpha': 32, 'beta': 0.1,
            'w4_format_at_start': W4_FORMAT,
        },
    )
    print(f'✓ Trackio run: {RUN_NAME}')
except Exception as e:
    TRACKIO_OK = False
    print(f'⚠️  Trackio init failed ({e!r}); training will use console logging only.')

In [ ]:
# 9) DPO training — Qwen-2.5-7B + LoRA r=32, optionally starting from W4 KTO adapter.
#
# Same review-fixed config defaults as W4 (cell 10):
#   - bf16 base loaded explicitly (P0 #1 fix)
#   - effective batch 128 (gradient_accumulation_steps=64)
#   - max_length=1536, max_prompt_length=1024
#   - save_steps=50 for OOM survivability
#   - per_device_eval_batch_size=4 to bound eval time
#
# DPO-specific:
#   - DPOConfig (not KTOConfig)
#   - beta=0.1 (KL penalty; same as KTO)
#   - learning_rate=5e-7 (TRL DPO default; aggressive enough for our LoRA)
#   - loss_type='sigmoid' (vanilla DPO; we expanded 1-pos-N-neg → pairs)
import torch, gc
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM
from trl import DPOTrainer, DPOConfig

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
HUB_REPO = f'orrimoch/recsys2026-{RUN_NAME}'
OUTPUT_DIR = f'/content/recsys2026/training_runs/{RUN_NAME}'

peft_config = LoraConfig(
    r=32, lora_alpha=32, lora_dropout=0.05,
    bias='none', task_type='CAUSAL_LM', target_modules='all-linear',
)

# Load base in bf16. If W4 KTO adapter is available, MERGE it into the base
# weights so the new S-DPO LoRA refines on top of an envelope-fluent model.
# (Alternative: load adapter as trainable; less clean. Merging avoids
# nested-PEFT confusion in TRL DPOTrainer.)
print(f'loading {MODEL_NAME} in bf16...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map='cuda',
)
if KTO_ADAPTER:
    print(f'loading + merging W4 KTO adapter from {KTO_ADAPTER}...')
    peft_model = PeftModel.from_pretrained(base_model, KTO_ADAPTER)
    base_model = peft_model.merge_and_unload()
    del peft_model
    gc.collect(); torch.cuda.empty_cache()
    print(f'✓ KTO weights merged; will train fresh LoRA r=32 on top')
else:
    print('⚠️  no W4 KTO adapter — starting fresh LoRA on un-trained Qwen-7B')

base_model.enable_input_require_grads()
print(f'✓ base ready ({base_model.num_parameters() / 1e9:.1f}B params)')

config = DPOConfig(
    output_dir=OUTPUT_DIR,

    # Hub push (W4 P1 fix mandate)
    push_to_hub=True,
    hub_model_id=HUB_REPO,
    hub_strategy='every_save',
    hub_private_repo=True,

    # DPO
    beta=0.1,
    loss_type='sigmoid',  # vanilla DPO; we expanded 1-pos-N-neg → pairs

    # Sequence length (W4 P1 fix: bumped to fit envelope + state + response)
    max_length=1536,
    max_prompt_length=1024,

    # Training — W5 P1 #6 fix: safer cold-start default (batch=1, accum=128).
    # The merge_and_unload step at training start adds ~10 GB transient VRAM
    # vs. W4's clean cold-start. Same effective batch (128), lower peak VRAM.
    # If A100-40GB has headroom after first checkpoint, can be bumped to (2, 64).
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=128,
    learning_rate=5e-7,  # TRL DPO default — preference-style aggressive enough
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    bf16=True,
    gradient_checkpointing=True,

    # Eval (W4 P1 fix: explicit eval batch)
    eval_strategy='steps',
    eval_steps=200,
    per_device_eval_batch_size=4,

    # Checkpointing (W4 P1 fix: earlier first save)
    save_strategy='steps',
    save_steps=50,
    save_total_limit=3,
    logging_steps=10,

    # Monitoring
    report_to='trackio' if TRACKIO_OK else 'none',
)

trainer = DPOTrainer(
    model=base_model,
    args=config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=peft_config,
)

print('🚀 Starting S-DPO training (~6 A100-hr expected)...')
print(f'   model: {MODEL_NAME}{"+W4-merged" if KTO_ADAPTER else ""}')
print(f'   adapter → {HUB_REPO}')
print(f'   effective batch: {config.per_device_train_batch_size * config.gradient_accumulation_steps}')
trainer.train()
print('✓ training complete')


In [ ]:
# 10) Push to Hub + Drive backup.
trainer.push_to_hub()
print(f'✓ adapter at https://huggingface.co/{HUB_REPO}')

import shutil
drive_dst = f'{DRIVE_BASE}/sdpo_runs/{RUN_NAME}'
shutil.copytree(OUTPUT_DIR, drive_dst, dirs_exist_ok=True)
print(f'✓ adapter mirrored to {drive_dst}')

In [ ]:
# 10b) DEPLOYMENT FIX (W5 review P0 #2):
# The W5 LoRA was trained on top of (Qwen-7B + W4-merged-base). It encodes
# a delta relative to that, NOT relative to raw Qwen-7B. Loading the W5
# adapter via vllm_model.LoRARequest on top of raw Qwen-7B produces garbage
# at inference because every neuron differs from what the adapter was
# trained against.
#
# Fix: after S-DPO training, MERGE the W5 LoRA into the (already-merged
# W4) base, then push the FULLY-MERGED 7B model to Hub as a standalone
# repo. Production config 210 references this merged model directly with
# `lora_path: null` — no adapter at inference.
#
# Cost: ~14 GB Hub upload (~3-5 min on Colab uplink). One-time per W5 run.
# Trade-off: vs. ~150 MB adapter — the storage cost is real but the
# correctness guarantee is worth it.

import gc
from peft import PeftModel

# Free the trainer (and its policy + reference + optimizer state) first.
del trainer
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f'free VRAM after trainer del: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')

# Re-load the base + W4-merged-weights + W5 adapter, then merge W5 in.
print(f'rebuilding inference stack: base + W4-merged + W5 LoRA…')
inf_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map='cuda',
)
if KTO_ADAPTER:
    pm_w4 = PeftModel.from_pretrained(inf_base, KTO_ADAPTER)
    inf_base = pm_w4.merge_and_unload()
    del pm_w4
    gc.collect(); torch.cuda.empty_cache()
    print('  ✓ W4 merged into base')

# Load the W5 adapter and merge it on top.
pm_w5 = PeftModel.from_pretrained(inf_base, OUTPUT_DIR)
fully_merged = pm_w5.merge_and_unload()
del pm_w5
gc.collect(); torch.cuda.empty_cache()
print('  ✓ W5 merged into (W4-merged) base — fully-merged 7B ready')

# Push the fully-merged model to Hub.
MERGED_REPO = f'orrimoch/recsys2026-{RUN_NAME}-merged'
print(f'pushing fully-merged 7B → {MERGED_REPO} (this may take a few minutes)...')
fully_merged.push_to_hub(MERGED_REPO, private=True, commit_message=f'W5 S-DPO merged (W4+W5) on Qwen-7B')
tok.push_to_hub(MERGED_REPO, private=True)  # tokenizer too — needed at inference
print(f'✓ deployment artifact at https://huggingface.co/{MERGED_REPO}')
print(f'  → use this as lm_type in config/210-responder-sdpo-qwen7b-devset.yaml')
print(f'  → leave lora_path: null (model is already fully merged)')


In [ ]:
# 11) Format-compliance check — same chat-template + r_format gate as W4.
# Free training state first to avoid a second full-7B load OOM (W4 P0 #4 fix).
import gc, sys, os, re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

del trainer
gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
print(f'free VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
inference_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map='cuda',
)
if KTO_ADAPTER:
    pm = PeftModel.from_pretrained(inference_base, KTO_ADAPTER)
    inference_base = pm.merge_and_unload()
    del pm
    gc.collect(); torch.cuda.empty_cache()
model = PeftModel.from_pretrained(inference_base, OUTPUT_DIR).eval()
print('✓ inference model ready (base + W4 merged + W5 adapter)')

# Same production chat-template path as W4 cell 12.
sys.path.insert(0, '/content/recsys2026/scripts')
from reward_fns import r_format, ENVELOPE

PROMPTS_DIR = '/content/recsys2026/music-crs-baselines/mcrs/system_prompts'
with open(f'{PROMPTS_DIR}/roleplay.txt', encoding='utf-8') as f:
    role_play = f.read()
with open(f'{PROMPTS_DIR}/response_generation_cot_user_state.txt', encoding='utf-8') as f:
    cot_prompt = f.read()
SYSTEM_PROMPT = role_play + '\n\n' + cot_prompt

USER_QUERY_RE = re.compile(r'^User query:\s*(.+?)(?=\nListener goal:|\nGoal category:|$)',
                            re.DOTALL | re.MULTILINE)
def extract_user_query(prompt_text):
    m = USER_QUERY_RE.search(prompt_text)
    return m.group(1).strip() if m else prompt_text[:200]

user_queries = [extract_user_query(eval_ds[i]['prompt']) for i in range(min(50, len(eval_ds)))]
n_strict = n_loose = 0
samples = []
for uq in user_queries:
    chat = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': uq},
    ]
    formatted = tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    enc = tok(formatted, return_tensors='pt', truncation=True, max_length=2048).to('cuda')
    with torch.no_grad():
        out_ids = model.generate(
            **enc, max_new_tokens=320, do_sample=False,
            pad_token_id=tok.pad_token_id or tok.eos_token_id,
        )
    text = tok.decode(out_ids[0, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    if r_format(text) == 1.0:
        n_strict += 1
    if ENVELOPE.search(text):
        n_loose += 1
    if len(samples) < 3:
        samples.append(text[:400])

compliance_strict = n_strict / len(user_queries)
compliance_loose = n_loose / len(user_queries)
print(f'\nFORMAT COMPLIANCE (post-S-DPO):')
print(f'  strict r_format:  {n_strict}/{len(user_queries)} = {compliance_strict:.1%}')
print(f'  loose ENVELOPE:   {n_loose}/{len(user_queries)} = {compliance_loose:.1%}')

if W4_FORMAT is not None:
    delta = compliance_strict - W4_FORMAT
    print(f'\n  Δ vs W4: {delta:+.1%}  (W4 was {W4_FORMAT:.1%})')

print('\nSample generations:')
for i, s in enumerate(samples, 1):
    print(f'\n--- sample {i} ---\n{s}')

print('\n' + '=' * 60)
print('W5 B2 GATE (plan §6.3 row B2):')
if compliance_strict >= 0.95:
    print(f'  PASS  strict r_format compliance {compliance_strict:.1%} ≥ 95%')
    print('  S-DPO successfully reinforced format. Proceed to W6 (Rank-GRPO).')
elif W4_FORMAT is not None and compliance_strict > W4_FORMAT:
    print(f'  IMPROVEMENT but no PASS: {compliance_strict:.1%} > W4 {W4_FORMAT:.1%} but < 95%.')
    print('  Options: (a) ship to W6 anyway, (b) more S-DPO epochs, (c) re-extract train state.')
else:
    print(f'  FAIL  S-DPO did not improve format ({compliance_strict:.1%} ≤ W4 {W4_FORMAT or 0:.1%})')
    print('  Likely diagnosis: data is too noisy / pairs not differentiated enough.')
    print('  Skip W5, ship W4 to W6 with the format gap acknowledged.')
print('=' * 60)

if TRACKIO_OK:
    trackio.log({
        'format_compliance_strict': compliance_strict,
        'format_compliance_loose': compliance_loose,
        'delta_vs_w4': compliance_strict - (W4_FORMAT or 0.0),
    })

In [ ]:
# 12) Persist gate result + finish Trackio.
import json
from datetime import date
result = {
    'stage': 'B2-S-DPO',
    'run_name': RUN_NAME,
    'date': date.today().isoformat(),
    'hub_model': HUB_REPO,
    'merged_hub_model': MERGED_REPO,  # P0 #2 deployment artifact
    'starting_adapter': KTO_ADAPTER,
    'w4_format': W4_FORMAT,
    'format_compliance_strict': compliance_strict,
    'format_compliance_loose': compliance_loose,
    'gate_passed': compliance_strict >= 0.95,
    'n_eval_samples': len(user_queries),
    'sample_outputs': samples,
    'config': {
        'method': 'S-DPO (vanilla TRL DPO on 1-pos-N-neg expanded pairs)',
        'effective_batch_size': 128,
        'lora_r': 32,
        'beta': 0.1,
        'lr': 5e-7,
    },
}
out_path = f'{DRIVE_BASE}/sdpo_runs/{RUN_NAME}/gate_result.json'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print(f'gate result → {out_path}')

if TRACKIO_OK:
    trackio.finish()
print('✓ done')


## After the run

**S-DPO PASSES (`strict ≥ 95%`):**
- Adapter is at `https://huggingface.co/{HUB_REPO}` (private).
- Run `colab/40_run_blindset_B.ipynb` (TBD) with `lora_path={HUB_REPO}` to check dev nDCG@20 no-regression.
- Proceed to W6 (Rank-GRPO).

**S-DPO IMPROVES but doesn't reach 95%:**
- Most likely: data noise + small model capacity. Increase `num_train_epochs=2` and re-run.
- Alternatively: pre-extract train states (notebook 22) so envelope augmentation has real state blocks instead of `(unknown)` — gives the model more discrimination signal.

**S-DPO doesn't improve over W4:**
- Likely: the 1-pos-N-neg expansion lost too much signal. The proper Plackett-Luce S-DPO would handle this, but requires a custom DPOTrainer subclass (~150 LOC). Document the gap, ship W4 to W6.

## Cost-saving knobs (for the user's noted scaling work)

1. **Reduce N_SESSIONS in cell 6** (currently 15000). Linear in compute. 5000 = 2 A100-hr, 1500 = ~30 min.
2. **Switch base model to Qwen-2.5-3B** (instead of 7B). Halves VRAM and ~2× speed. Trade-off: response quality cap is lower.
3. **Drop one of the 4 negative types** in `build_sdpo_dataset.py PERTURBATION_VARIANTS`. Reduces total pairs by ~25%.
4. **Reduce `gradient_accumulation_steps` from 64 → 16** (effective batch 32 instead of 128). Same wallclock but training will be noisier.
5. **Reduce LoRA `r` from 32 → 16**. ~50% fewer trainable params. Plan §6.3.1 mandates 32, but 16 is the TRL default.